In [ ]:
# ================================================================
# CONSIDERACIONES TÉCNICAS TRANSVERSALES
# ================================================================
# 1. Idempotencia → ya se cumple con mode("overwrite")
# 2. Pruebas de calidad → 5 validaciones automatizadas
# ================================================================

from pyspark.sql.functions import (
    col, lit, current_timestamp, count, when,
    round as spark_round, sum as spark_sum
)
from pyspark.sql import Row
from datetime import datetime
import traceback

CATALOG = "Prueba Tecnica Dataknow"
BRONZE_LH = "LH_BRONZE_PRUEBA"
SILVER_LH = "LH_SILVER_PRUEBA"
GOLD_LH = "LH_GOLD_PRUEBA"
SCHEMA = "dbo"

def brz(t): return f"`{CATALOG}`.{BRONZE_LH}.{SCHEMA}.brz_{t}"
def slv(t): return f"`{CATALOG}`.{SILVER_LH}.{SCHEMA}.slv_{t}"
def gold(t): return f"`{CATALOG}`.{GOLD_LH}.{SCHEMA}.{t}"
def err(t): return f"`{CATALOG}`.{SILVER_LH}.{SCHEMA}.err_{t}"

StatementMeta(, e3701201-facb-4471-bccb-4bc31af6ebf0, 10, Finished, Available, Finished, False)

In [ ]:
# ================================================================
# 1. IDEMPOTENCIA
# ================================================================
# Ya se cumple porque todas las escrituras usan:
#   .mode("overwrite") + .option("overwriteSchema", "true")
#
# Esto significa que si corres el pipeline 1, 2 o 10 veces
# sobre los mismos datos, el resultado es siempre el mismo.
# No se acumulan duplicados ni se alteran los conteos.
#
# Verificación:
print("=" * 60)
print("1️⃣  IDEMPOTENCIA")
print("=" * 60)

conteos = {}
tablas_gold = [
    "dim_clientes", "dim_productos", "dim_geografia", "dim_canal",
    "fact_transacciones", "fact_cartera", "fact_rentabilidad_cliente",
    "kpi_diario_cartera"
]

for t in tablas_gold:
    c = spark.sql(f"SELECT COUNT(*) as c FROM {gold(t)}").collect()[0]["c"]
    conteos[t] = c
    print(f"  {t}: {c:,} registros")

print("\n  ✅ Todas las tablas usan mode('overwrite').")
print("  ✅ Ejecutar N veces produce siempre estos mismos conteos.")


StatementMeta(, e3701201-facb-4471-bccb-4bc31af6ebf0, 11, Finished, Available, Finished, False)

1️⃣  IDEMPOTENCIA


  dim_clientes: 10,000 registros


  dim_productos: 50 registros


  dim_geografia: 200 registros


  dim_canal: 200 registros


  fact_transacciones: 499,500 registros


  fact_cartera: 30,000 registros


  fact_rentabilidad_cliente: 26,481 registros
  kpi_diario_cartera: 19,158 registros

  ✅ Todas las tablas usan mode('overwrite').
  ✅ Ejecutar N veces produce siempre estos mismos conteos.


In [ ]:

# ================================================================
# 2. PRUEBAS DE CALIDAD DE DATOS (5 validaciones)
# ================================================================
print(f"\n{'=' * 60}")
print("3️⃣  PRUEBAS DE CALIDAD DE DATOS")
print("=" * 60)

resultados_calidad = []

def registrar_prueba(nombre, descripcion, tabla, condicion_sql, umbral_pct=100):
    """
    Ejecuta una prueba de calidad y registra si pasa o falla.
    umbral_pct: porcentaje mínimo de registros que deben cumplir la condición.
    """
    total = spark.sql(f"SELECT COUNT(*) as c FROM {tabla}").collect()[0]["c"]
    cumplen = spark.sql(f"SELECT COUNT(*) as c FROM {tabla} WHERE {condicion_sql}").collect()[0]["c"]
    pct = round((cumplen / total * 100), 2) if total > 0 else 0
    pasa = pct >= umbral_pct

    resultado = "✅ PASA" if pasa else "❌ FALLA"
    print(f"\n  {resultado} — {nombre}")
    print(f"    {descripcion}")
    print(f"    Tabla: {tabla}")
    print(f"    Cumplen: {cumplen:,}/{total:,} ({pct}%) — Umbral: {umbral_pct}%")

    resultados_calidad.append(Row(
        prueba=nombre,
        descripcion=descripcion,
        tabla=tabla.split(".")[-1],
        total_registros=total,
        registros_ok=cumplen,
        registros_falla=total - cumplen,
        pct_ok=pct,
        umbral_pct=float(umbral_pct),
        resultado="PASA" if pasa else "FALLA",
        timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ))


# --- PRUEBA 1: Unicidad de llave primaria ---
registrar_prueba(
    nombre="DQ-001: Unicidad de llave primaria en dim_clientes",
    descripcion="Cada id_cli debe aparecer exactamente una vez",
    tabla=gold("dim_clientes"),
    condicion_sql="""id_cli IN (
        SELECT id_cli FROM """ + gold("dim_clientes") + """
        GROUP BY id_cli HAVING COUNT(*) = 1
    )""",
    umbral_pct=100
)

# --- PRUEBA 2: No nulos en campos críticos ---
registrar_prueba(
    nombre="DQ-002: Completitud de fact_transacciones",
    descripcion="id_mov, id_cli, vr_mov y fec_mov no deben ser nulos",
    tabla=gold("fact_transacciones"),
    condicion_sql="id_mov IS NOT NULL AND id_cli IS NOT NULL AND vr_mov IS NOT NULL AND fec_mov IS NOT NULL",
    umbral_pct=100
)

# --- PRUEBA 3: Integridad referencial ---
registrar_prueba(
    nombre="DQ-003: Integridad referencial fact_transacciones → dim_clientes",
    descripcion="Todo id_cli en transacciones debe existir en dim_clientes",
    tabla=gold("fact_transacciones"),
    condicion_sql="""id_cli IN (
        SELECT id_cli FROM """ + gold("dim_clientes") + """
    )""",
    umbral_pct=100
)

# --- PRUEBA 4: Rango válido de valores ---
registrar_prueba(
    nombre="DQ-004: Montos positivos en fact_transacciones",
    descripcion="vr_mov debe ser mayor que cero",
    tabla=gold("fact_transacciones"),
    condicion_sql="vr_mov > 0",
    umbral_pct=99  # permitimos un 1% de margen
)

# --- PRUEBA 5: Consistencia de provision vs cartera ---
registrar_prueba(
    nombre="DQ-005: Provisión no excede saldo capital",
    descripcion="provision_estimada debe ser menor o igual a sdo_capital",
    tabla=gold("fact_cartera"),
    condicion_sql="provision_estimada <= sdo_capital",
    umbral_pct=100
)


# Guardar resultados de calidad
df_calidad = spark.createDataFrame(resultados_calidad)
df_calidad.withColumn("_fec_ejecucion", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{CATALOG}`.{GOLD_LH}.{SCHEMA}.reporte_pruebas_calidad")

print(f"\n{'=' * 60}")
print("📊 RESUMEN DE PRUEBAS DE CALIDAD")
print(f"{'=' * 60}")
spark.sql(f"""
    SELECT prueba, resultado, registros_ok, registros_falla, 
           CONCAT(pct_ok, '%') as cumplimiento,
           CONCAT(umbral_pct, '%') as umbral
    FROM `{CATALOG}`.{GOLD_LH}.{SCHEMA}.reporte_pruebas_calidad
""").show(truncate=False)

total_pasan = len([r for r in resultados_calidad if r.resultado == "PASA"])
total_fallan = len([r for r in resultados_calidad if r.resultado == "FALLA"])
print(f"  ✅ Pasan: {total_pasan}/5")
if total_fallan > 0:
    print(f"  ❌ Fallan: {total_fallan}/5")

print(f"\n{'=' * 60}")
print("✅ CONSIDERACIONES TRANSVERSALES COMPLETADAS")
print(f"{'=' * 60}")
print(f"""
Resumen:
  1. Idempotencia    → mode('overwrite') en todas las capas
  2. Pruebas calidad → 5 validaciones en reporte_pruebas_calidad

Tablas generadas:
  • reporte_pruebas_calidad  → {GOLD_LH}
""")

StatementMeta(, e3701201-facb-4471-bccb-4bc31af6ebf0, 13, Finished, Available, Finished, False)


3️⃣  PRUEBAS DE CALIDAD DE DATOS

  ✅ PASA — DQ-001: Unicidad de llave primaria en dim_clientes
    Cada id_cli debe aparecer exactamente una vez
    Tabla: `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.dim_clientes
    Cumplen: 10,000/10,000 (100.0%) — Umbral: 100%



  ✅ PASA — DQ-002: Completitud de fact_transacciones
    id_mov, id_cli, vr_mov y fec_mov no deben ser nulos
    Tabla: `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.fact_transacciones
    Cumplen: 499,500/499,500 (100.0%) — Umbral: 100%



  ✅ PASA — DQ-003: Integridad referencial fact_transacciones → dim_clientes
    Todo id_cli en transacciones debe existir en dim_clientes
    Tabla: `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.fact_transacciones
    Cumplen: 499,500/499,500 (100.0%) — Umbral: 100%



  ✅ PASA — DQ-004: Montos positivos en fact_transacciones
    vr_mov debe ser mayor que cero
    Tabla: `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.fact_transacciones
    Cumplen: 499,498/499,500 (100.0%) — Umbral: 99%



  ✅ PASA — DQ-005: Provisión no excede saldo capital
    provision_estimada debe ser menor o igual a sdo_capital
    Tabla: `Prueba Tecnica Dataknow`.LH_GOLD_PRUEBA.dbo.fact_cartera
    Cumplen: 30,000/30,000 (100.0%) — Umbral: 100%



📊 RESUMEN DE PRUEBAS DE CALIDAD


+----------------------------------------------------------------+---------+------------+---------------+------------+------+
|prueba                                                          |resultado|registros_ok|registros_falla|cumplimiento|umbral|
+----------------------------------------------------------------+---------+------------+---------------+------------+------+
|DQ-003: Integridad referencial fact_transacciones → dim_clientes|PASA     |499500      |0              |100.0%      |100.0%|
|DQ-002: Completitud de fact_transacciones                       |PASA     |499500      |0              |100.0%      |100.0%|
|DQ-005: Provisión no excede saldo capital                       |PASA     |30000       |0              |100.0%      |100.0%|
|DQ-001: Unicidad de llave primaria en dim_clientes              |PASA     |10000       |0              |100.0%      |100.0%|
|DQ-004: Montos positivos en fact_transacciones                  |PASA     |499498      |2              |100.0%      |